# Baseline CellTypist predictions: interactive label matrices

This notebook visualizes the **unmasked baseline** predictions for the selected Human Immune Health Atlas cells. It uses the existing `iha_expression_prepared.h5ad`, `baseline_predictions.csv`, and `baseline_decision_matrix.csv` files. If either prediction file is missing, it runs CellTypist on the unmasked prepared data and saves both outputs.

It creates three interactive HTML matrices: **mapped atlas types**, **unmapped atlas types**, and **all atlas types**. Rows remain the original atlas `cell_type`; columns are observed CellTypist predictions. Classes with zero predictions in a view are filtered out. Colors show percentages within each original atlas type; hover reveals counts and both row and column percentages.

Large colored rectangles group labels by CellTypist high hierarchy. A dark solid outline marks a link from the current `dataset_label_mapping.csv`; an orange dashed outline marks an exact shared Cell Ontology ID. These outlines show candidate relationships and do not certify prediction correctness. Original labels are not overwritten or treated as error-free ground truth.

This is a descriptive label-distribution analysis, not a simple accuracy report. Gene masking is not used here.


In [1]:
from pathlib import Path

import anndata as ad
import celltypist
from celltypist import models
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.colors as pc

# Change this folder if running the notebook on another computer.
DATA_ROOT = Path("/Users/artsvetkova/Studies/_JKU/3rd_semester/AI and Visualization/UE")
BASE_DIR = DATA_ROOT / "iha_celltypist"
PREPARED_H5AD = BASE_DIR / "iha_expression_prepared.h5ad"
PREDICTIONS_CSV = BASE_DIR / "baseline_predictions.csv"
DECISION_MATRIX_CSV = BASE_DIR / "baseline_decision_matrix.csv"
MODEL_NAME = "Immune_All_Low.pkl"

LABEL_MAPPING_CSV = BASE_DIR / "dataset_label_mapping.csv"
MODEL_CROSSWALK_CSV = BASE_DIR / "model_celltypist_crosswalk_candidates.csv"
OUTPUT_DIR = BASE_DIR / "label_audit" / "interactive_matrices"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CELL_KEY = "soma_joinid"
ATLAS_COL = "cell_type"
ATLAS_CL_COL = "cell_type_ontology_term_id"
PRED_COL = "predicted_labels"
HIGH_GROUP_ORDER = [
    "B cells", "Plasma cells", "T cells", "Double-negative thymocytes",
    "Double-positive thymocytes", "ILC", "pDC", "DC", "Monocytes",
    "Mono-mac", "Macrophages", "Myelocytes", "HSC/MPP", "Erythroid",
    "Early MK", "Megakaryocyte precursor", "Megakaryocytes/platelets",
    "Cycling cells", "Other",
]
BLOCK_COLORS = pc.qualitative.Dark24

for required in (PREPARED_H5AD, LABEL_MAPPING_CSV, MODEL_CROSSWALK_CSV):
    if not required.exists():
        raise FileNotFoundError(f"Required input not found: {required}")

print(f"Unmasked prepared data: {PREPARED_H5AD}")
print(f"Interactive outputs: {OUTPUT_DIR}")


/Users/artsvetkova/miniconda3/envs/sc_immune/lib/python3.10/site-packages/celltypist/classifier.py:11: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  from scanpy import __version__ as scv


Unmasked prepared data: /Users/artsvetkova/Studies/_JKU/3rd_semester/AI and Visualization/UE/iha_celltypist/iha_expression_prepared.h5ad
Interactive outputs: /Users/artsvetkova/Studies/_JKU/3rd_semester/AI and Visualization/UE/iha_celltypist/label_audit/interactive_matrices


## 1. Load metadata and align the baseline outputs

Existing predictions are reused, so CellTypist will not rerun unless one of its saved CSV files is missing. Every join is checked against `soma_joinid`.

In [2]:
def read_cell_table(csv_path):
    table = pd.read_csv(csv_path, dtype=str)
    key = CELL_KEY if CELL_KEY in table.columns else table.columns[0]
    if table[key].isna().any() or table[key].duplicated().any():
        raise ValueError(f"{csv_path.name} has missing or duplicate cell IDs.")
    table = table.set_index(key)
    table.index = table.index.astype(str)
    table.index.name = CELL_KEY
    return table


def run_and_save_baseline():
    model = models.Model.load(MODEL_NAME)
    input_data = ad.read_h5ad(PREPARED_H5AD)
    result = celltypist.annotate(input_data, model=model, majority_voting=False)
    result.predicted_labels.to_csv(PREDICTIONS_CSV, index_label=CELL_KEY)
    result.decision_matrix.to_csv(DECISION_MATRIX_CSV, index_label=CELL_KEY)
    print("Ran CellTypist on the unmasked prepared data and saved baseline outputs.")


if not PREDICTIONS_CSV.exists() or not DECISION_MATRIX_CSV.exists():
    run_and_save_baseline()

adata = ad.read_h5ad(PREPARED_H5AD, backed="r")
try:
    required_obs = [CELL_KEY, ATLAS_COL, ATLAS_CL_COL]
    missing = [column for column in required_obs if column not in adata.obs.columns]
    if missing:
        raise KeyError(f"Prepared H5AD is missing obs columns: {missing}")
    metadata = adata.obs[required_obs].copy()
finally:
    adata.file.close()

numeric_ids = pd.to_numeric(metadata[CELL_KEY], errors="raise")
if numeric_ids.isna().any():
    raise ValueError("Prepared H5AD contains missing soma_joinid values.")
metadata[CELL_KEY] = numeric_ids.astype("int64").astype(str)
if metadata[CELL_KEY].duplicated().any():
    raise ValueError("Prepared H5AD contains duplicate soma_joinid values.")
metadata = metadata.set_index(CELL_KEY, drop=False)

pred_table = read_cell_table(PREDICTIONS_CSV)
if PRED_COL not in pred_table.columns:
    raise KeyError(f"{PREDICTIONS_CSV.name} must contain {PRED_COL!r}.")
score_table = read_cell_table(DECISION_MATRIX_CSV)
expected_ids = metadata.index
for name, table in (("predictions", pred_table), ("decision matrix", score_table)):
    missing = expected_ids.difference(table.index)
    extra = table.index.difference(expected_ids)
    if len(missing) or len(extra):
        raise ValueError(f"{name} IDs differ from the H5AD: {len(missing):,} missing; {len(extra):,} extra.")

comparison = metadata.join(pred_table[[PRED_COL]], how="left", validate="one_to_one")
if comparison[PRED_COL].isna().any():
    raise ValueError("Some cells have no baseline prediction.")
comparison[ATLAS_COL] = comparison[ATLAS_COL].astype(str)
comparison[PRED_COL] = comparison[PRED_COL].astype(str)
comparison[ATLAS_CL_COL] = comparison[ATLAS_CL_COL].astype("string")

print(f"Cells aligned: {len(comparison):,}")
print(f"Original atlas labels: {comparison[ATLAS_COL].nunique():,}")
print(f"Observed CellTypist classes: {comparison[PRED_COL].nunique():,}")


Cells aligned: 54,225
Original atlas labels: 36
Observed CellTypist classes: 61


## 2. Define mapped subsets and hierarchy-based ordering

`mapped` and `unmapped` indicate only whether an atlas label occurs in the current manual mapping CSV. The matrix still displays the original `cell_type` values. For visual ordering only, unmapped atlas types are anchored to their most frequent baseline prediction; this does not make that prediction a verified match.

In [3]:
label_mapping = pd.read_csv(LABEL_MAPPING_CSV, dtype=str).dropna(
    subset=["atlas_cell_type", "new_dataset_label"]
)
if label_mapping["atlas_cell_type"].duplicated().any():
    raise ValueError("dataset_label_mapping.csv has duplicate atlas_cell_type rows.")
atlas_to_manual_label = dict(zip(label_mapping["atlas_cell_type"], label_mapping["new_dataset_label"]))
comparison["mapping_status"] = np.where(
    comparison[ATLAS_COL].isin(atlas_to_manual_label), "mapped", "unmapped"
)

model_crosswalk = pd.read_csv(MODEL_CROSSWALK_CSV, dtype=str).dropna(subset=["model_label"])
for column in ("High-hierarchy cell types", "Cell Ontology ID"):
    if column not in model_crosswalk.columns:
        raise KeyError(f"Model crosswalk is missing {column!r}.")
model_groups = {}
for label, group in model_crosswalk.groupby("model_label", sort=False):
    values = [x for x in group["High-hierarchy cell types"].dropna().astype(str).unique() if x.strip()]
    model_groups[str(label)] = values[0] if values else "Other"

observed_classes = set(comparison[PRED_COL])
all_model_classes = [column for column in score_table.columns if column != PRED_COL]
never_predicted = sorted(set(all_model_classes) - observed_classes)
pd.DataFrame({"model_class_never_predicted": never_predicted}).to_csv(
    OUTPUT_DIR / "model_classes_never_predicted.csv", index=False
)
print(f"Model classes never predicted in this baseline: {len(never_predicted)}")

# Global dominant prediction is used only as a visual ordering anchor for unmapped types.
dominant_prediction = comparison.groupby(ATLAS_COL, observed=True)[PRED_COL].agg(
    lambda values: values.value_counts().index[0]
).to_dict()

def atlas_group(atlas_label):
    anchor = atlas_to_manual_label.get(atlas_label, dominant_prediction.get(atlas_label, ""))
    return model_groups.get(anchor, "Other")

atlas_groups = {label: atlas_group(label) for label in comparison[ATLAS_COL].unique()}
group_rank = {group: i for i, group in enumerate(HIGH_GROUP_ORDER)}

# A strict CL outline is used only when a model class has one unambiguous CL ID.
def normalize_cl(value):
    if pd.isna(value) or str(value).strip().casefold() in {"", "nan", "none", "<na>"}:
        return ""
    return str(value).strip()

model_cl = {}
for label, group in model_crosswalk.groupby("model_label", sort=False):
    terms = {normalize_cl(value) for value in group["Cell Ontology ID"]}
    terms.discard("")
    if len(terms) == 1:
        model_cl[str(label)] = next(iter(terms))

manual_pairs = set(atlas_to_manual_label.items())
strict_cl_pairs = set()
for _, row in comparison[[ATLAS_COL, ATLAS_CL_COL]].drop_duplicates().iterrows():
    atlas_term = normalize_cl(row[ATLAS_CL_COL])
    if atlas_term:
        for model_label, model_term in model_cl.items():
            if atlas_term == model_term:
                strict_cl_pairs.add((str(row[ATLAS_COL]), model_label))

print(f"Manual mapping links available to outline: {len(manual_pairs)}")
print(f"Unambiguous exact CL links available to outline: {len(strict_cl_pairs)}")


Model classes never predicted in this baseline: 37
Manual mapping links available to outline: 19
Unambiguous exact CL links available to outline: 20


## 3. Build the filtered interactive matrices

Each view contains only atlas rows and model columns with at least one cell in that subset. CSVs keep exact counts and row percentages. In the HTML, hover over a cell for its count, share within the atlas row, and share within the model-class column. Colored rectangles mark high-hierarchy blocks; small dark and orange outlines show the mapping links described above.

In [4]:
def make_interactive_matrix(data, subset_name):
    if data.empty:
        print(f"No cells in {subset_name}; skipping this view.")
        return None
    counts = pd.crosstab(data[ATLAS_COL].astype(str), data[PRED_COL].astype(str))
    counts = counts.loc[counts.sum(axis=1) > 0, counts.sum(axis=0) > 0]
    if counts.empty:
        print(f"No observed label pairs in {subset_name}; skipping this view.")
        return None

    row_pct = counts.div(counts.sum(axis=1), axis=0) * 100
    col_pct = counts.div(counts.sum(axis=0), axis=1) * 100
    model_order = sorted(
        counts.columns,
        key=lambda label: (
            group_rank.get(model_groups.get(label, "Other"), len(group_rank)),
            model_groups.get(label, "Other"), label,
        ),
    )
    model_rank = {label: i for i, label in enumerate(model_order)}

    def row_key(label):
        anchor = atlas_to_manual_label.get(label, dominant_prediction.get(label, ""))
        return (
            group_rank.get(atlas_groups.get(label, "Other"), len(group_rank)),
            model_rank.get(anchor, len(model_rank)),
            0 if label in atlas_to_manual_label else 1,
            -int(counts.loc[label].sum()),
            label,
        )

    atlas_order = sorted(counts.index, key=row_key)
    counts = counts.loc[atlas_order, model_order]
    row_pct = row_pct.loc[atlas_order, model_order]
    col_pct = col_pct.loc[atlas_order, model_order]
    counts.to_csv(OUTPUT_DIR / f"{subset_name}_counts.csv")
    row_pct.to_csv(OUTPUT_DIR / f"{subset_name}_row_percent.csv")

    shape = counts.shape
    count_values = counts.to_numpy(dtype=float)
    row_totals = np.broadcast_to(counts.sum(axis=1).to_numpy(dtype=float)[:, None], shape)
    row_values = row_pct.to_numpy(dtype=float)
    col_totals = np.broadcast_to(counts.sum(axis=0).to_numpy(dtype=float)[None, :], shape)
    col_values = col_pct.to_numpy(dtype=float)
    customdata = np.stack([count_values, row_totals, row_values, col_totals, col_values], axis=-1)

    fig = go.Figure(go.Heatmap(
        z=row_values,
        x=model_order,
        y=atlas_order,
        customdata=customdata,
        colorscale="Blues",
        zmin=0,
        zmax=100,
        colorbar=dict(title="% within<br>atlas type"),
        hovertemplate=(
            "Original atlas type: %{y}<br>CellTypist class: %{x}<br>"
            "Cells in this pair: %{customdata[0]:,.0f}<br>"
            "Cells with this atlas type: %{customdata[1]:,.0f}<br>"
            "Share within atlas type: %{customdata[2]:.1f}%<br>"
            "Cells assigned this model class: %{customdata[3]:,.0f}<br>"
            "Share of model class from this atlas type: %{customdata[4]:.1f}%<extra></extra>"
        ),
    ))

    # Colored rectangles outline contiguous blocks in the high-hierarchy ordering.
    present_groups = []
    for group in HIGH_GROUP_ORDER:
        x_pos = [i for i, label in enumerate(model_order) if model_groups.get(label, "Other") == group]
        y_pos = [i for i, label in enumerate(atlas_order) if atlas_groups.get(label, "Other") == group]
        if not x_pos or not y_pos:
            continue
        present_groups.append(group)
        color = BLOCK_COLORS[(len(present_groups) - 1) % len(BLOCK_COLORS)]
        fig.add_shape(
            type="rect", x0=min(x_pos) - 0.5, x1=max(x_pos) + 0.5,
            y0=min(y_pos) - 0.5, y1=max(y_pos) + 0.5,
            xref="x", yref="y",
            line=dict(color=color, width=2), fillcolor="rgba(0,0,0,0)", layer="above",
        )

    row_pos = {label: i for i, label in enumerate(atlas_order)}
    col_pos = {label: i for i, label in enumerate(model_order)}
    outline_specs = [
        (manual_pairs, "#222222", "solid", "Current manual link"),
        (strict_cl_pairs, "#e6550d", "dash", "Exact shared CL ID"),
    ]
    legend_added = set()
    for pairs, color, dash, legend_label in outline_specs:
        for atlas_label, model_label in pairs:
            if atlas_label not in row_pos or model_label not in col_pos:
                continue
            fig.add_shape(
                type="rect",
                x0=col_pos[model_label] - 0.47,
                x1=col_pos[model_label] + 0.47,
                y0=row_pos[atlas_label] - 0.47,
                y1=row_pos[atlas_label] + 0.47,
                xref="x", yref="y",
                line=dict(color=color, width=2, dash=dash),
                fillcolor="rgba(0,0,0,0)", layer="above",
            )
            if legend_label not in legend_added:
                fig.add_trace(go.Scatter(
                    x=[None], y=[None], mode="markers",
                    marker=dict(size=9, color=color, symbol="square-open", line=dict(color=color, width=2)),
                    name=legend_label, hoverinfo="skip", showlegend=True,
                ))
                legend_added.add(legend_label)

    fig.update_layout(
        title=dict(text=f"Unmasked baseline prediction distribution: {subset_name} atlas labels", x=0.5, y=0.98),
        template="simple_white",
        width=max(1250, 500 + 34 * len(model_order)),
        height=max(780, 350 + 30 * len(atlas_order)),
        margin=dict(t=185, l=270, r=220, b=210),
        legend=dict(
            orientation="h",
            x=0.5,
            xanchor="center",
            y=1.14,
            yanchor="bottom",
            font=dict(size=10),
            itemsizing="constant",
            tracegroupgap=8,
        ),
        annotations=[dict(
            text="Color: % within atlas type · Large colored frames: hierarchy blocks · Dark solid: manual mapping · Orange dashed: exact shared CL ID (may outline a zero-count cell)",
            x=0.5, y=1.055, xref="paper", yref="paper", showarrow=False,
        )],
    )
    fig.update_xaxes(title="CellTypist predicted class", categoryorder="array", categoryarray=model_order, tickangle=-50, automargin=True)
    fig.update_yaxes(title="Original atlas cell_type", categoryorder="array", categoryarray=atlas_order, autorange="reversed", automargin=True)

    html_path = OUTPUT_DIR / f"{subset_name}_interactive.html"
    fig.write_html(
        html_path,
        include_plotlyjs=True,
        config={"scrollZoom": True, "displaylogo": False, "toImageButtonOptions": {"format": "png", "scale": 3}},
    )
    print(f"{subset_name}: {len(atlas_order)} atlas labels × {len(model_order)} observed model classes")
    print(f"Saved: {html_path}")
    return fig, counts, row_pct


matrix_results = {}
for subset_name, subset in [
    ("mapped", comparison.loc[comparison["mapping_status"] == "mapped"]),
    ("unmapped", comparison.loc[comparison["mapping_status"] == "unmapped"]),
    ("all", comparison),
]:
    matrix_results[subset_name] = make_interactive_matrix(subset, subset_name)

print()
print("Finished. Open the HTML files from:", OUTPUT_DIR.resolve())


mapped: 19 atlas labels × 54 observed model classes
Saved: /Users/artsvetkova/Studies/_JKU/3rd_semester/AI and Visualization/UE/iha_celltypist/label_audit/interactive_matrices/mapped_interactive.html
unmapped: 17 atlas labels × 48 observed model classes
Saved: /Users/artsvetkova/Studies/_JKU/3rd_semester/AI and Visualization/UE/iha_celltypist/label_audit/interactive_matrices/unmapped_interactive.html
all: 36 atlas labels × 61 observed model classes
Saved: /Users/artsvetkova/Studies/_JKU/3rd_semester/AI and Visualization/UE/iha_celltypist/label_audit/interactive_matrices/all_interactive.html

Finished. Open the HTML files from: /Users/artsvetkova/Studies/_JKU/3rd_semester/AI and Visualization/UE/iha_celltypist/label_audit/interactive_matrices
